In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
AZURE_SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT")
AZURE_SEARCH_API_KEY = os.getenv("AZURE_SEARCH_API_KEY")

print("✅ Environment variables loaded")
print("OpenAI Endpoint:", AZURE_OPENAI_ENDPOINT)
print("Search Endpoint:", AZURE_SEARCH_ENDPOINT)

✅ Environment variables loaded
OpenAI Endpoint: https://rag-chatbotrix-openai.openai.azure.com/
Search Endpoint: https://rag-chatbotrix-search.search.windows.net


In [2]:
from openai import AzureOpenAI

openai_client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-02-01"
)

def embed_question(question):
    response = openai_client.embeddings.create(
        input=question,
        model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT
    )
    return response.data[0].embedding

question = "What is the return policy?"
question_vector = embed_question(question)

print(f"✅ Question embedded")
print(f"Vector length: {len(question_vector)}")

✅ Question embedded
Vector length: 1536


In [3]:
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery
from azure.core.credentials import AzureKeyCredential

search_client = SearchClient(
    endpoint=AZURE_SEARCH_ENDPOINT,
    index_name="rag-chatbotrix-index",
    credential=AzureKeyCredential(AZURE_SEARCH_API_KEY)
)

def retrieve_chunks(question, top_k=3):
    question_vector = embed_question(question)
    
    vector_query = VectorizedQuery(
        vector=question_vector,
        k_nearest_neighbors=top_k,
        fields="embedding"
    )
    
    results = search_client.search(
        search_text=question,
        vector_queries=[vector_query],
        select=["content"],
        top=top_k
    )
    
    chunks = []
    for result in results:
        chunks.append(result["content"])
    
    return chunks

chunks = retrieve_chunks("What is the return policy?")

print(f"✅ Retrieved {len(chunks)} chunk(s)\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk)
    print()

✅ Retrieved 1 chunk(s)

--- Chunk 1 ---
Contoso Electronics offers a range of laptops and accessories. Our return policy allows returns within 30 days of purchase. All products come with a 1-year manufacturer warranty. For support, customers can contact us at support@contoso.com. Our store is open Monday to Friday, 9am to 6pm.



In [4]:
test_questions = [
    "What are the store hours?",
    "How do I contact support?",
    "What warranty do products come with?"
]

for question in test_questions:
    print(f"❓ Question: {question}")
    chunks = retrieve_chunks(question)
    print(f"✅ Retrieved {len(chunks)} chunk(s)")
    print(f"Answer chunk: {chunks[0][:100]}...")
    print()

❓ Question: What are the store hours?
✅ Retrieved 1 chunk(s)
Answer chunk: Contoso Electronics offers a range of laptops and accessories. Our return policy allows returns with...

❓ Question: How do I contact support?
✅ Retrieved 1 chunk(s)
Answer chunk: Contoso Electronics offers a range of laptops and accessories. Our return policy allows returns with...

❓ Question: What warranty do products come with?
✅ Retrieved 1 chunk(s)
Answer chunk: Contoso Electronics offers a range of laptops and accessories. Our return policy allows returns with...



In [5]:
def ask_gpt(question):
    # Step 1: retrieve relevant chunks
    chunks = retrieve_chunks(question)
    
    # Step 2: combine chunks into context
    context = "\n\n".join(chunks)
    
    # Step 3: build prompt
    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant. Answer the question using ONLY the context provided. If the answer is not in the context, say 'I don't know'."
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }
    ]
    
    # Step 4: call GPT-4o
    response = openai_client.chat.completions.create(
        model=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT"),
        messages=messages,
        temperature=0.2
    )
    
    return response.choices[0].message.content

# Test it
answer = ask_gpt("What is the return policy?")
print(f"❓ Question: What is the return policy?")
print(f"🤖 Answer: {answer}")

❓ Question: What is the return policy?
🤖 Answer: The return policy allows returns within 30 days of purchase.


In [6]:
test_questions = [
    "What is the return policy?",
    "What are the store hours?",
    "How do I contact support?",
    "What warranty do products come with?",
    "Do you sell phones?"
]

for question in test_questions:
    answer = ask_gpt(question)
    print(f"❓ {question}")
    print(f"🤖 {answer}")
    print()

❓ What is the return policy?
🤖 The return policy allows returns within 30 days of purchase.

❓ What are the store hours?
🤖 The store is open Monday to Friday, 9am to 6pm.

❓ How do I contact support?
🤖 You can contact support by emailing support@contoso.com.

❓ What warranty do products come with?
🤖 All products come with a 1-year manufacturer warranty.

❓ Do you sell phones?
🤖 I don't know.

